# Laboratorio — Robot de entregas en un almacén

A partir de la **imagen**, construye el MDP y resuélvelo con **Value Iteration** y **Policy Iteration**.

![Mundo del ejercicio](https://drive.google.com/uc?export=view&id=1_sJaD57gHuiz1joEgl4B-u0aDy8jtMDo)



## Convención y notación

$$
s=(row,col)
$$

$$
T(s,a,s')=P(s'\mid s,a)
$$

$$
R(s)
$$

Para Value Iteration:

$$
V_{k+1}(s)
=
R(s)
+
\gamma
\max_a
\sum_{s'}T(s,a,s')V_k(s')
$$

Para Policy Evaluation:

$$
V_{k+1}^{\pi}(s)
=
R(s)
+
\gamma
\sum_{s'}T(s,\pi(s),s')V_k^\pi(s')
$$

### Acciones

```python
UP    = (-1, 0)
DOWN  = ( 1, 0)
LEFT  = ( 0,-1)
RIGHT = ( 0, 1)
```



## Reglas del mundo

El grid tiene **5 filas × 6 columnas**.

### Estados especiales

A partir de la imagen identifica:

- `START`
- estanterías / paredes;
- zona de entrega `+10` (**terminal**);
- estación de carga `+2` (**terminal**);
- peligro mortal `-10` (**terminal**);
- peligros `-3` (**no terminales**);
- celdas de piso resbaloso.

### Recompensa

Usamos la convención del notebook de clase, es decir, **\(R(s)\)**:

- entrega: `+10`;
- carga: `+2`;
- peligro mortal: `-10`;
- peligro: `-3`;
- cualquier otro estado transitable: `-1` (costo por paso).

### Dinámica

La transición depende del **estado actual**:

**Piso normal**

$$
P(\text{dirección elegida})=0.90
$$

$$
P(\text{desviación izquierda})=0.05
$$

$$
P(\text{desviación derecha})=0.05
$$

**Piso resbaloso**

$$
P(\text{dirección elegida})=0.60
$$

$$
P(\text{desviación izquierda})=0.20
$$

$$
P(\text{desviación derecha})=0.20
$$

Si el movimiento sale del grid o golpea una estantería, el robot **permanece en el mismo estado**.

Usa:

$$
\gamma=0.9,\qquad \theta=10^{-4}
$$



## Parte 1 — Modela el MDP

Completa la clase `WarehouseMDP`.

La parte importante no es escribir muchas líneas de código: es traducir correctamente la imagen a:

- estados;
- acciones;
- recompensas;
- terminales;
- obstáculos;
- tipos de piso;
- función de transición.


In [1]:
import numpy as np

class WarehouseMDP:
    def __init__(self):
        self.height = 5
        self.width = 6

        # --- A partir de la imagen ---
        self.start = (0, 0)

        self.walls = {
            (0, 3), (1, 1), (2, 4), (4, 2)
        }

        self.slippery_states = {
            (1, 2), (2, 1), (3, 3)
        }

        self.terminal_states = {
            (0, 5): 10.0,   # zona de entrega
            (2, 2): 2.0,    # estacion de carga
            (3, 5): -10.0,  # peligro mortal
        }

        self.danger_states = {
            (1, 4): -3.0,
            (4, 1): -3.0,
        }

        self.living_reward = -1.0
        self.gamma = 0.9

        self.actions = [
            (-1, 0),  # UP
            ( 1, 0),  # DOWN
            ( 0,-1),  # LEFT
            ( 0, 1),  # RIGHT
        ]

    def is_valid_state(self, state):
        r, c = state
        if not (0 <= r < self.height and 0 <= c < self.width):
            return False
        if state in self.walls:
            return False
        return True

    def states(self):
        return [
            (r, c)
            for r in range(self.height)
            for c in range(self.width)
            if (r, c) not in self.walls
        ]

    def is_terminal(self, state):
        return state in self.terminal_states

    def get_reward(self, state):
        if state in self.terminal_states:
            return self.terminal_states[state]
        if state in self.danger_states:
            return self.danger_states[state]
        return self.living_reward

    def get_transition_probs(self, state, action):
        """
        Devuelve:
            [(next_state, probability), ...]
        """
        # Un estado terminal es absorbente: te quedas ahi con prob. 1
        if self.is_terminal(state):
            return [(state, 1.0)]

        # Las probabilidades dependen del tipo de piso del ESTADO ORIGEN
        if state in self.slippery_states:
            p_intended, p_side = 0.60, 0.20
        else:
            p_intended, p_side = 0.90, 0.05

        dr, dc = action
        left_action  = (-dc, dr)   # rotacion 90 grados antihoraria
        right_action = ( dc, -dr)  # rotacion 90 grados horaria

        outcomes = [
            (action,       p_intended),
            (left_action,  p_side),
            (right_action, p_side),
        ]

        # Acumulamos probabilidad por next_state (varios intentos pueden
        # rebotar al mismo estado si hay pared/borde)
        probs = {}
        for act, p in outcomes:
            nr, nc = state[0] + act[0], state[1] + act[1]
            next_state = (nr, nc) if self.is_valid_state((nr, nc)) else state
            probs[next_state] = probs.get(next_state, 0.0) + p

        return list(probs.items())



### Validación mínima del modelo

Antes de implementar Bellman, valida primero el MDP.


In [2]:
grid = WarehouseMDP()

S = grid.states()
print("Número de estados:", len(S))

# Cada distribución T(s,a,·) debe sumar 1.
for s in S:
    for a in grid.actions:
        transitions = grid.get_transition_probs(s, a)
        total = sum(p for _, p in transitions)
        assert abs(total - 1.0) < 1e-12

print("✓ Todas las distribuciones de transición suman 1.")


Número de estados: 26
✓ Todas las distribuciones de transición suman 1.



## Parte 2 — Value Iteration

Implementa:

$$
V_{k+1}(s)
=
R(s)+\gamma\max_a
\sum_{s'}T(s,a,s')V_k(s')
$$


In [3]:
def expected_next_value(grid, state, action, V):
    # sum_{s'} T(s,a,s') V(s')
    total = 0.0
    for next_state, prob in grid.get_transition_probs(state, action):
        total += prob * V[next_state]
    return total


def value_iteration(grid, threshold=1e-4, max_iter=10_000):
    S = grid.states()
    V = {s: 0.0 for s in S}

    for i in range(max_iter):
        delta = 0.0
        V_new = {}
        for s in S:
            if grid.is_terminal(s):
                V_new[s] = grid.get_reward(s)
            else:
                q_values = [
                    grid.get_reward(s) + grid.gamma * expected_next_value(grid, s, a, V)
                    for a in grid.actions
                ]
                V_new[s] = max(q_values)
            delta = max(delta, abs(V_new[s] - V[s]))
        V = V_new
        if delta < threshold:
            return V, i + 1

    return V, max_iter


def extract_policy(grid, V):
    # pi*(s) = argmax_a sum T(s,a,s') V(s')
    policy = {}
    for s in grid.states():
        if grid.is_terminal(s):
            continue
        best_action = max(
            grid.actions,
            key=lambda a: expected_next_value(grid, s, a, V)
        )
        policy[s] = best_action
    return policy



## Parte 4 — Visualización de resultados


In [4]:
ARROWS = {
    (-1, 0): "↑",
    ( 1, 0): "↓",
    ( 0,-1): "←",
    ( 0, 1): "→",
}

def print_values(grid, V):
    for r in range(grid.height):
        row = []
        for c in range(grid.width):
            s = (r, c)
            if s in grid.walls:
                row.append("  WALL  ")
            else:
                row.append(f"{V[s]:+7.3f}")
        print(" | ".join(row))


def print_policy(grid, policy):
    for r in range(grid.height):
        row = []
        for c in range(grid.width):
            s = (r, c)

            if s in grid.walls:
                row.append(" # ")
            elif grid.is_terminal(s):
                reward = grid.get_reward(s)
                row.append(f"{reward:+.0f}")
            else:
                row.append(f" {ARROWS[policy[s]]} ")

        print(" | ".join(row))


In [5]:
# VALUE ITERATION
V_vi, n_vi = value_iteration(grid)
pi_vi = extract_policy(grid, V_vi)

print("=== VALUE ITERATION ===")
print("Iteraciones:", n_vi)
print("\nValores:")
print_values(grid, V_vi)
print("\nPolítica:")
print_policy(grid, pi_vi)


=== VALUE ITERATION ===
Iteraciones: 20

Valores:
 -2.575 |  -1.679 |  -0.652 |   WALL   |  +7.607 | +10.000
 -2.193 |   WALL   |  +0.560 |  +2.104 |  +3.670 |  +7.607
 -1.229 |  -0.063 |  +2.000 |  +0.832 |   WALL   |  +5.673
 -1.770 |  -0.731 |  +0.552 |  -0.782 |  -1.837 | -10.000
 -2.732 |  -3.890 |   WALL   |  -1.837 |  -2.692 |  -3.802

Política:
 →  |  →  |  ↓  |  #  |  →  | +10
 ↓  |  #  |  ↓  |  →  |  →  |  ↑ 
 →  |  →  | +2 |  ↑  |  #  |  ↑ 
 →  |  →  |  ↑  |  ↑  |  ←  | -10
 ↑  |  ↑  |  #  |  ↑  |  ←  |  ← 



## Parte 5 — Interpreta la política

Antes de cambiar parámetros, responde:

1. Desde `START`, ¿el robot busca la **entrega +10** o prefiere la **estación de carga +2**?
2. ¿Por qué una recompensa menor podría ser óptima?
3. ¿En qué estados el piso resbaloso cambia la decisión?
4. ¿Qué papel cumple el costo por paso `-1`?
5. ¿Por qué \(T(s,a,s')\) ya no puede implementarse con las mismas probabilidades para todos los estados?

### Experimento A — Menos costo por paso

Cambia:

```python
living_reward = -0.1
```

Predice la política **antes de ejecutar**.

R/la recompensa que recibe el robot en cada paso normal (no terminal). Antes cada movimiento "costaba" -1, ahora casi no cuesta nada (-0.1).
Prueba si a urgencia por terminar rápido es lo que empuja al robot a conformarse con la carga (+2) en vez de ir por la entrega (+10). El costo por paso es el "precio del tiempo": cuanto más alto, más penaliza tomar rutas largas, aunque terminen mejor. Ya política cambió por lo que el robot ahora sí va por la entrega (+10), aceptando una ruta de 7 pasos en vez de 4. Confirma la hipótesis: al quitar casi todo el costo de caminar, ya no hay razón para conformarse con la recompensa cercana y menor.

### Experimento B — Piso muy resbaloso

Cambia la probabilidad de movimiento deseado del piso resbaloso:

```python
0.60 → 0.40
```

y reparte el restante entre las dos desviaciones.

La política no cambió sigue prefiriendo la carga (+2) por el mismo camino corto. En esa celda, da igual para dónde intentes moverte (a la carga o a la entrega): si te resbalas, casi siempre terminas cerca de la carga de todos modos. Entonces subir la probabilidad de resbalón no perjudica más a un camino que al otro, perjudica a los dos por igual.

Como no hay un "lado perdedor" que se vea más afectado, la comparación entre ir a carga o ir a entrega no cambia, y por eso la política se queda igual.

### Experimento C — Más paciencia

Cambia:

```python
gamma = 0.99
```

¿La política valora más la recompensa `+10` distante?

la política cambió a la entrega (+10), igual que en el Experimento A. Tiene sentido porque el costo por paso y el descuento actúan sobre el mismo mecanismo: ambos determinan cuánto "pesa" la distancia. Bajar el costo por paso (A) o subir la paciencia (C) tienen el mismo efecto práctico aquí.

### Bonus

Encuentra aproximadamente el valor de `living_reward` a partir del cual la política desde `START` cambia entre:

- ir a carga `+2`;
- intentar llegar a entrega `+10`.

El cambio ocurre alrededor de living_reward ≈ -0.80. Es el punto de equilibrio entre "ganar +2 rápido y seguro" y "ganar +10 lejano pero costoso en pasos". O sea: si caminar cuesta menos de -0.8 (por ejemplo -0.5), el robot va por la entrega. Si cuesta más de -0.8 (por ejemplo -1.0), el robot prefiere quedarse con la carga, que es más segura y cercana.


## Respuestas — Parte 5

**1. Desde `START`, ¿el robot busca la entrega +10 o la carga +2?**

Con los parametros base (`living_reward = -1.0`, `gamma = 0.9`, piso normal 0.90/0.05/0.05), el robot prefiere la estacion de carga (+2). La politica optima desde `START` es: `START(0,0) -> (0,1) -> (0,2) -> (1,2) -> (2,2)=CARGA`, es decir, baja por la columna 2 hasta la carga en 4 pasos.

**2. ¿Por que una recompensa menor (+2) puede ser optima?**

Porque lo que se maximiza no es la recompensa terminal aislada, sino el retorno descontado total, que incluye el costo acumulado de -1 por cada paso dado y el riesgo de pasar cerca de celdas de peligro (-3) o del peligro mortal (-10). El camino hacia la entrega (+10) es mas largo y pasa cerca de la celda de peligro (1,4) y de zonas resbalosas, mientras que el camino a la carga es corto y mas seguro. El valor neto de "+2 rapido y seguro" termina superando a "+10 lejano y riesgoso".

**3. ¿En que estados el piso resbaloso cambia la decision?**

Los estados resbalosos son `(1,2)`, `(2,1)` y `(3,3)`. En `(1,2)`, que esta justo en la ruta corta hacia la carga, la mayor probabilidad de desviacion (0.20 en vez de 0.05) no alcanza a hacer que valga la pena rodear, porque las desviaciones ahi llevan a celdas igualmente seguras. En cambio, cerca de `(3,3)` y cualquier celda adyacente a peligros, una probabilidad de desviacion mayor si castiga mas fuerte una politica que pase demasiado cerca del peligro mortal (3,5) o de los -3, empujando esas rutas a mantenerse mas alejadas del peligro.

**4. ¿Que papel cumple el costo por paso -1?**

Es lo que le da urgencia al robot. Sin ese costo (o con un costo casi nulo, ver Experimento A), no hay presion por terminar rapido, y el robot esta mas dispuesto a tomar rutas largas para llegar a una recompensa terminal mayor. Con el costo de -1 completo, cada paso adicional "cuesta", asi que rutas cortas a recompensas menores pueden ganarle a rutas largas hacia recompensas mayores.

**5. ¿Por que T(s,a,s') ya no puede ser igual para todos los estados?**

Porque la probabilidad de "desviacion" (0.05 vs 0.20) depende de si el estado de origen es piso resbaloso o no. El kernel de transicion deja de ser homogeneo en el espacio: para construir `T(s,a,*)` primero hay que consultar `state in self.slippery_states` y elegir el conjunto de probabilidades correspondiente antes de repartir la masa de probabilidad entre la accion elegida y sus dos desviaciones.
